In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading Prepared Data...")
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')

y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_val = pd.read_csv('../data/processed/y_val.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape}")

In [ ]:
print("Training Logistic Regression (Baseline)...")
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Predict on Validation Set
y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)[:, 1]

In [ ]:
acc = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred)
rec = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
auc = roc_auc_score(y_val, y_val_prob)

baseline_metrics = {
    'model_name': 'Logistic Regression (Baseline)',
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1_score': float(f1),
    'roc_auc': float(auc)
}

os.makedirs('../models', exist_ok=True)
with open('../models/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=4)

print("Validation Metrics:")
print(json.dumps(baseline_metrics, indent=2))

# Save Model
joblib.dump(model, '../models/baseline_model.pkl')
print("Baseline model saved to models/baseline_model.pkl")

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Active (0)', 'Churned (1)'], 
            yticklabels=['Active (0)', 'Churned (1)'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Baseline Confusion Matrix (Validation)')
os.makedirs('../visualizations/model', exist_ok=True)
plt.savefig('../visualizations/model/baseline_confusion_matrix.png')
plt.show()
print("Confusion matrix saved.")